# Dog Breed Classification & "Which Dog Are You?"
**Dr. Yoram Segal** | **Hadar Wayne**

## How to Run
1. Runtime -> Change runtime type -> **A100 or T4 GPU**
2. Run **Cell 1** (setup) -- authorize Drive when prompted
3. Run **Cell 2** (keep-alive) -- prevents Colab disconnection
4. Run **Cell 3** (train) -- auto-skips already-completed models
5. Run **Cell 4** (evaluate + export) -- generates final JSON

## Disconnect Recovery
If Colab disconnects, just reconnect and re-run cells 1-4.
All progress is saved to Google Drive. Completed models are auto-skipped.
Individual transfer learning stages are also auto-skipped if completed.


In [ ]:
#@title CELL 1: COMPLETE SETUP (run this first)
#@markdown This cell sets up everything: imports, Drive, data, models, training functions.
#@markdown On first run it downloads data (~10 min). On re-run it skips to "SETUP COMPLETE".

import os,sys,time,json,random,shutil,requests
from pathlib import Path
from collections import defaultdict
from io import BytesIO
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.notebook import tqdm
import torch
import torch.nn as nn
from torch.utils.data import DataLoader,WeightedRandomSampler
from torchvision import transforms,datasets,models
from sklearn.metrics import accuracy_score,top_k_accuracy_score,confusion_matrix

DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')
else:
    print('WARNING: No GPU! Runtime -> Change runtime type -> GPU')

from google.colab import drive
drive.mount('/content/drive')
DR='/content/drive/MyDrive/L41_DogBreeds'
for d in ['','/data/raw','/data/processed/train','/data/processed/val',
          '/results/models','/results/history','/results/graphs','/results/tables']:
    os.makedirs(f'{DR}{d}',exist_ok=True)
DD=Path(f'{DR}/data'); RD=DD/'raw'; TD=DD/'processed'/'train'; VD=DD/'processed'/'val'
RR=Path(f'{DR}/results'); HD=RR/'history'; MD=RR/'models'

# === KAGGLE ===
os.system('pip install -q kagglehub')
os.environ['KAGGLE_API_TOKEN']='KGAT_c5800001ede614248a98d33ff496d977'
if not (RD/'labels.csv').exists():
    import kagglehub
    print('Downloading from Kaggle...')
    p=kagglehub.competition_download('dog-breed-identification')
    for i in Path(p).iterdir():
        d=RD/i.name
        if i.is_dir():
            if not d.exists(): print(f'  Copying {i.name}/...'); shutil.copytree(i,d)
        else: shutil.copy2(i,d)
    print('Dataset on Drive!')
else:
    print('Dataset already on Drive')
ldf=pd.read_csv(RD/'labels.csv')
print(f'{len(ldf)} images, {ldf["breed"].nunique()} breeds')

# === ORGANIZE ===
SD=42; IS=224; IM=[.485,.456,.406]; IST=[.229,.224,.225]; BS=32; NW=2
if not TD.exists() or len(list(TD.iterdir()))<50:
    print('Organizing into breed folders...')
    bi=defaultdict(list)
    for _,r in ldf.iterrows():
        p=RD/'train'/f"{r['id']}.jpg"
        if p.exists(): bi[r['breed']].append(p)
    for d in [TD,VD]:
        if d.exists(): shutil.rmtree(d)
        d.mkdir(parents=True)
    random.seed(SD); tt=tv=0
    for b,imgs in sorted(bi.items()):
        random.shuffle(imgs); si=max(1,int(len(imgs)*.8))
        for sn,sl,sd in [('t',imgs[:si],TD),('v',imgs[si:],VD)]:
            bd=sd/b; bd.mkdir(parents=True,exist_ok=True)
            for ip in sl:
                try:
                    Image.open(ip).convert('RGB').resize((IS,IS),Image.LANCZOS).save(bd/ip.name,'JPEG',quality=95)
                    if sn=='t': tt+=1
                    else: tv+=1
                except: pass
    print(f'{len(bi)} breeds, {tt} train, {tv} val')
else:
    print(f'Already organized: {sum(len(list(d.glob("*.jpg"))) for d in TD.iterdir() if d.is_dir())} train')

# === TRANSFORMS & DATALOADERS ===
def gtf(sz=224, tr=True):
    if tr:
        return transforms.Compose([
            transforms.RandomResizedCrop(sz, scale=(.8,1.)),
            transforms.RandomHorizontalFlip(.5),
            transforms.RandomRotation(15),
            transforms.ColorJitter(.2,.2,.2,.1),
            transforms.ToTensor(),
            transforms.Normalize(IM, IST)])
    return transforms.Compose([
        transforms.Resize(int(sz*1.14)),
        transforms.CenterCrop(sz),
        transforms.ToTensor(),
        transforms.Normalize(IM, IST)])

def gdl(sz=224):
    tds=datasets.ImageFolder(str(TD), transform=gtf(sz,True))
    vds=datasets.ImageFolder(str(VD), transform=gtf(sz,False))
    tg=np.array(tds.targets)
    cc=np.maximum(np.bincount(tg, minlength=len(tds.classes)).astype(float), 1.)
    cw=torch.FloatTensor(len(tg)/(len(tds.classes)*cc))
    sw=[cw[l].item() for _,l in tds.samples]
    tl=DataLoader(tds, batch_size=BS, sampler=WeightedRandomSampler(sw,len(sw),replacement=True),
                  num_workers=NW, pin_memory=True, drop_last=True)
    vl=DataLoader(vds, batch_size=BS, shuffle=False, num_workers=NW, pin_memory=True)
    print(f'DataLoaders: {len(tds)} train, {len(vds)} val, {len(tds.classes)} classes')
    return tl, vl, tds.classes, cw

trl,vll,CN,CW = gdl()
NC = len(CN)

# === MODELS ===
class SCNN(nn.Module):
    def __init__(self, nc=120):
        super().__init__()
        self.f = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(True), nn.MaxPool2d(2))
        self.c = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(128,512), nn.ReLU(True), nn.Dropout(.5), nn.Linear(512,nc))
    def forward(self, x):
        return self.c(self.f(x))

def gm(n, nc=120, pt=True):
    if n=='simple_cnn': return SCNN(nc)
    if n=='alexnet':
        m=models.alexnet(weights=models.AlexNet_Weights.DEFAULT if pt else None)
        m.classifier[6]=nn.Linear(m.classifier[6].in_features, nc)
    elif n=='vgg16':
        m=models.vgg16(weights=models.VGG16_Weights.DEFAULT if pt else None)
        m.classifier[6]=nn.Linear(m.classifier[6].in_features, nc)
    elif n=='inception':
        m=models.inception_v3(weights=models.Inception_V3_Weights.DEFAULT if pt else None)
        m.fc=nn.Linear(m.fc.in_features, nc)
        if m.AuxLogits is not None:
            m.AuxLogits.fc=nn.Linear(m.AuxLogits.fc.in_features, nc)
    elif n=='resnet50':
        m=models.resnet50(weights=models.ResNet50_Weights.DEFAULT if pt else None)
        m.fc=nn.Linear(m.fc.in_features, nc)
    elif n=='mobilenet':
        m=models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT if pt else None)
        m.classifier[1]=nn.Linear(m.classifier[1].in_features, nc)
    else: raise ValueError(n)
    print(f'{n}: {sum(p.numel() for p in m.parameters()):,} params')
    return m

MN = ['simple_cnn','alexnet','vgg16','inception','resnet50','mobilenet']

# === TRAINING FUNCTIONS ===
def te(mo, ld, cr, op):
    mo.train(); ls=co=to=0
    for im,la in tqdm(ld, desc='Train', leave=False):
        im,la = im.to(DEVICE), la.to(DEVICE)
        op.zero_grad(); o=mo(im)
        if isinstance(o, tuple) and len(o)==2:
            loss=cr(o[0],la)+.4*cr(o[1],la); lg=o[0]
        elif isinstance(o, tuple):
            loss=cr(o[0],la); lg=o[0]
        else:
            loss=cr(o,la); lg=o
        loss.backward(); op.step()
        ls+=loss.item()*im.size(0); co+=lg.argmax(1).eq(la).sum().item(); to+=la.size(0)
    return ls/to, co/to

@torch.no_grad()
def ve(mo, ld, cr):
    mo.eval(); ls=co=to=0
    for im,la in tqdm(ld, desc='Val', leave=False):
        im,la = im.to(DEVICE), la.to(DEVICE)
        o=mo(im)
        if isinstance(o, tuple): o=o[0]
        loss=cr(o,la)
        ls+=loss.item()*im.size(0); co+=o.argmax(1).eq(la).sum().item(); to+=la.size(0)
    return ls/to, co/to

def dt(mo, tl, vl, ep, lr, w, nm, st=''):
    mo=mo.to(DEVICE)
    cr=nn.CrossEntropyLoss(weight=w.to(DEVICE))
    ps=[p for p in mo.parameters() if p.requires_grad]
    if not ps:
        print('WARNING: 0 trainable params!')
        return {'best_val_acc':0,'training_time_seconds':0,'epochs':[],'train_loss':[],'train_acc':[],'val_loss':[],'val_acc':[],'best_epoch':0,'model':nm,'stage':st,'lr':lr}
    op=torch.optim.Adam(ps, lr=lr)
    sc=torch.optim.lr_scheduler.ReduceLROnPlateau(op, patience=3, factor=.1)
    h={'model':nm,'stage':st,'lr':lr,'epochs':[],'train_loss':[],'train_acc':[],'val_loss':[],'val_acc':[],'best_val_acc':0,'best_epoch':0,'training_time_seconds':0}
    lb=f'{nm}[{st}]' if st else nm
    print(f'{lb} LR={lr} Ep={ep} Params={sum(p.numel() for p in ps):,}')
    t0=time.time(); pa=0
    for e in range(1, ep+1):
        tl_,ta = te(mo,tl,cr,op)
        vl_,va = ve(mo,vl,cr)
        sc.step(vl_)
        h['epochs'].append(e); h['train_loss'].append(tl_); h['train_acc'].append(ta)
        h['val_loss'].append(vl_); h['val_acc'].append(va)
        print(f'  E{e}/{ep} T:{ta:.4f}({tl_:.4f}) V:{va:.4f}({vl_:.4f}) LR:{op.param_groups[0]["lr"]:.1e}')
        if va > h['best_val_acc']:
            h['best_val_acc']=va; h['best_epoch']=e; pa=0
            sn=f'best_{nm}_{st}.pth' if st else f'best_{nm}.pth'
            torch.save(mo.state_dict(), MD/sn)
        else:
            pa+=1
        if pa>=5: print('  EarlyStop'); break
    h['training_time_seconds'] = time.time()-t0
    hn=f'{nm}_{st}_history.json' if st else f'{nm}_history.json'
    with open(HD/hn, 'w') as f: json.dump(h, f, indent=2)
    print(f'  Best:{h["best_val_acc"]:.4f}(ep{h["best_epoch"]}) {h["training_time_seconds"]:.0f}s')
    return h

def fa(m):
    for p in m.parameters(): p.requires_grad=False

def ut(m, fr):
    ps=list(m.parameters()); cu=len(ps)-max(1,int(len(ps)*fr))
    for i,p in enumerate(ps): p.requires_grad=(i>=cu)

def uc(m, n):
    a={'alexnet':'classifier','vgg16':'classifier','inception':'fc','resnet50':'fc','mobilenet':'classifier'}.get(n)
    if a and hasattr(m,a):
        for p in getattr(m,a).parameters(): p.requires_grad=True

TLS = {
    'stage1': {'n':'FeatureExtract', 'lr':1e-3, 'ep':10, 'f':0.},
    'stage2': {'n':'PartialFT', 'lr':1e-4, 'ep':10, 'f':.25},
    'stage3': {'n':'FullFT', 'lr':1e-5, 'ep':10, 'f':1.},
}

def dtl(mo, nm, tl, vl, w):
    rs={}
    for sk,cf in TLS.items():
        # Check if this stage already completed (resume support)
        hist_path = HD/f'{nm}_{sk}_history.json'
        if hist_path.exists():
            with open(hist_path) as f: prev=json.load(f)
            if prev.get('best_val_acc',0) > 0:
                print(f'\n  {sk}: Already done (acc={prev["best_val_acc"]:.4f}), skipping!')
                rs[sk]=prev['best_val_acc']
                # Load weights from this stage to continue to next
                wp=MD/f'best_{nm}_{sk}.pth'
                if wp.exists(): mo.load_state_dict(torch.load(wp, map_location=DEVICE, weights_only=True))
                continue
        print(f'\n{"="*40} {sk}:{cf["n"]} {"="*40}')
        if cf['f']==0: fa(mo); uc(mo,nm)
        else: ut(mo, cf['f'])
        h=dt(mo, tl, vl, cf['ep'], cf['lr'], w, nm, sk)
        rs[sk]=h['best_val_acc']
    bs=max(rs, key=rs.get)
    bp=MD/f'best_{nm}_{bs}.pth'
    if bp.exists():
        mo.load_state_dict(torch.load(bp, map_location=DEVICE, weights_only=True))
        torch.save(mo.state_dict(), MD/f'best_{nm}.pth')
    print(f'TL {nm}: {rs} Best:{bs}({rs[bs]:.4f})')
    return rs

def is_done(nm):
    """Check if a model is already fully trained (has best weights on Drive)."""
    return (MD/f'best_{nm}.pth').exists()

def pdb(pa, mo, k=3):
    mo.eval().to(DEVICE)
    tf=gtf(224, tr=False)
    t=tf(Image.open(pa).convert('RGB')).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        o=mo(t)
        if isinstance(o, tuple): o=o[0]
        pr=torch.softmax(o,1).squeeze()
    tp,ti=pr.topk(k)
    return [{'breed':CN[i.item()], 'confidence':round(p.item(),4)} for p,i in zip(tp,ti)]

def dli(url, pa, sz=224):
    try:
        r=requests.get(url, headers={'User-Agent':'Mozilla/5.0'}, timeout=15)
        if r.status_code==200 and 'image' in r.headers.get('content-type',''):
            Image.open(BytesIO(r.content)).convert('RGB').resize((sz,sz),Image.LANCZOS).save(pa,'JPEG',quality=95)
            return True
    except: pass
    return False

# Show what's already trained
print('\n--- Training Status (from Drive) ---')
for n in MN:
    if is_done(n):
        print(f'  {n}: DONE (weights on Drive)')
    else:
        # Check partial stages
        stages_done = [s for s in ['stage1','stage2','stage3'] if (HD/f'{n}_{s}_history.json').exists()]
        if stages_done:
            print(f'  {n}: PARTIAL ({", ".join(stages_done)} done)')
        else:
            print(f'  {n}: NOT STARTED')

print(f'\nSETUP COMPLETE | {DEVICE} | {NC} classes | {len(trl.dataset)} train')


In [ ]:
#@title CELL 2: KEEP-ALIVE (prevents Colab disconnection)
#@markdown Clicks the connect button every 60s to prevent timeout.

from IPython.display import display, Javascript
js_code = """
function ClickConnect(){
    console.log("Keeping Colab alive...");
    var btn = document.querySelector("colab-connect-button");
    if (btn) btn.click();
}
setInterval(ClickConnect, 60000);
console.log("Keep-alive started");
"""
display(Javascript(js_code))
print("Keep-alive activated! Colab will stay connected.")


In [ ]:
#@title CELL 2: TRAIN ALL MODELS (auto-skips already completed ones)
#@markdown Each model checks Drive for existing weights before training.
#@markdown If disconnected, just re-run Cell 1 then this cell -- it resumes automatically.

random.seed(SD); np.random.seed(SD); torch.manual_seed(SD)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SD)

# --- 1/6 Simple CNN ---
if is_done('simple_cnn'):
    print('=== 1/6 Simple CNN: ALREADY DONE, skipping ===')
else:
    print('=== 1/6 Simple CNN ===')
    dt(SCNN(NC), trl, vll, 15, 1e-3, CW, 'simple_cnn')
torch.cuda.empty_cache()

# --- 2/6 AlexNet ---
if is_done('alexnet'):
    print('\n=== 2/6 AlexNet: ALREADY DONE, skipping ===')
else:
    print('\n=== 2/6 AlexNet ===')
    m=gm('alexnet',NC); dtl(m,'alexnet',trl,vll,CW); del m
torch.cuda.empty_cache()

# --- 3/6 VGG-16 ---
if is_done('vgg16'):
    print('\n=== 3/6 VGG-16: ALREADY DONE, skipping ===')
else:
    print('\n=== 3/6 VGG-16 ===')
    m=gm('vgg16',NC); dtl(m,'vgg16',trl,vll,CW); del m
torch.cuda.empty_cache()

# --- 4/6 Inception ---
if is_done('inception'):
    print('\n=== 4/6 Inception: ALREADY DONE, skipping ===')
else:
    print('\n=== 4/6 Inception (299x299) ===')
    tl9,vl9,_,_=gdl(sz=299)
    m=gm('inception',NC); dtl(m,'inception',tl9,vl9,CW); del m
torch.cuda.empty_cache()

# --- 5/6 ResNet-50 ---
if is_done('resnet50'):
    print('\n=== 5/6 ResNet-50: ALREADY DONE, skipping ===')
else:
    print('\n=== 5/6 ResNet-50 ===')
    m=gm('resnet50',NC); dtl(m,'resnet50',trl,vll,CW); del m
torch.cuda.empty_cache()

# --- 6/6 MobileNet ---
if is_done('mobilenet'):
    print('\n=== 6/6 MobileNet: ALREADY DONE, skipping ===')
else:
    print('\n=== 6/6 MobileNet ===')
    m=gm('mobilenet',NC); dtl(m,'mobilenet',trl,vll,CW); del m
torch.cuda.empty_cache()

print('\n=== ALL TRAINING COMPLETE ===')
print('Models saved to Google Drive. Run Cell 3 for evaluation and export.')


In [ ]:
#@title CELL 3: EVALUATE + EXPERIMENTS + PLOTS + RESULTS EXPORT

# === EVALUATE ALL MODELS ===
@torch.no_grad()
def evm(mo, ld, nm):
    mo.eval().to(DEVICE)
    ap=[]; al=[]; apr=[]
    for im,la in tqdm(ld, desc=f'Eval {nm}'):
        o=mo(im.to(DEVICE))
        if isinstance(o, tuple): o=o[0]
        pr=torch.softmax(o,1).cpu().numpy()
        ap.extend(o.argmax(1).cpu().numpy()); al.extend(la.numpy()); apr.extend(pr)
    ap,al,apr = np.array(ap), np.array(al), np.array(apr)
    t1=accuracy_score(al,ap)
    t5=top_k_accuracy_score(al,apr,k=5,labels=list(range(NC)))
    print(f'{nm}: Top1={t1:.4f}({t1*100:.1f}%) Top5={t5:.4f}({t5*100:.1f}%)')
    return {'model':nm, 'top1':t1, 'top5':t5, 'preds':ap, 'labels':al, 'probs':apr}

AR={}
for n in MN:
    bp=MD/f'best_{n}.pth'
    if not bp.exists(): print(f'Skip {n}'); continue
    m=gm(n, NC, pt=False)
    m.load_state_dict(torch.load(bp, map_location=DEVICE, weights_only=True))
    ld = vl9 if n=='inception' else vll
    AR[n] = evm(m, ld, n)
    del m; torch.cuda.empty_cache()
print(f'Evaluated {len(AR)} models')

# === EXPERIMENTS ===
adf = pd.DataFrame()
cdf = pd.DataFrame()

if AR:
    bn = max(AR, key=lambda k: AR[k]['top1'])
    print(f'\nBest model: {bn} ({AR[bn]["top1"]:.1%})')
    bm = gm(bn, NC, pt=False)
    bm.load_state_dict(torch.load(MD/f'best_{bn}.pth', map_location=DEVICE, weights_only=True))
    bm.eval().to(DEVICE)

    # Animals
    print('\n--- Animal Experiment ---')
    AD = Path('/content/test_animals')
    au = {
        'horse':['https://images.unsplash.com/photo-1553284965-83fd3e82fa5a?w=300'],
        'zebra':['https://images.unsplash.com/photo-1501706362039-c06b2d715385?w=300'],
        'cat':['https://images.unsplash.com/photo-1514888286974-6c03e2ca1dba?w=300',
               'https://images.unsplash.com/photo-1573865526739-10659fec78a5?w=300'],
        'donkey':['https://images.unsplash.com/photo-1585336261022-680e295ce3fe?w=300'],
        'rabbit':['https://images.unsplash.com/photo-1585110396000-c9ffd4e4b308?w=300'],
        'fox':['https://images.unsplash.com/photo-1516934024742-b461fba47600?w=300'],
        'wolf':['https://images.unsplash.com/photo-1564466809058-bf4114d55352?w=300'],
        'bear':['https://images.unsplash.com/photo-1530595467537-0b5996c41f2d?w=300'],
        'lion':['https://images.unsplash.com/photo-1546182990-dffeafbe841d?w=300'],
        'cow':['https://images.unsplash.com/photo-1570042225831-d98fa7577f1e?w=300'],
    }
    ar=[]
    for a,urls in au.items():
        ad=AD/a; ad.mkdir(parents=True, exist_ok=True)
        for i,u in enumerate(urls):
            p=ad/f'{a}_{i+1:02d}.jpg'
            if not p.exists(): dli(u, p)
            if p.exists():
                try:
                    pr=pdb(str(p), bm)
                    ar.append({'animal':a, 'breed_1':pr[0]['breed'], 'conf_1':pr[0]['confidence']})
                    print(f'  {a}: {pr[0]["breed"]} ({pr[0]["confidence"]:.1%})')
                except Exception as e:
                    print(f'  {a}: {e}')
    adf = pd.DataFrame(ar)
    if len(adf)>0:
        adf.to_csv(RR/'tables'/'non_dog_predictions.csv', index=False)

    # Celebrities
    print('\n--- Celebrity Experiment ---')
    HH = Path('/content/test_humans'); HH.mkdir(parents=True, exist_ok=True)
    fu = {
        'person_01':'https://images.unsplash.com/photo-1507003211169-0a1dd7228f2d?w=300',
        'person_02':'https://images.unsplash.com/photo-1494790108377-be9c29b29330?w=300',
        'person_03':'https://images.unsplash.com/photo-1500648767791-00dcc994a43e?w=300',
        'person_04':'https://images.unsplash.com/photo-1438761681033-6461ffad8d80?w=300',
        'person_05':'https://images.unsplash.com/photo-1472099645785-5658abf4ff4e?w=300',
        'person_06':'https://images.unsplash.com/photo-1544005313-94ddf0286df2?w=300',
        'person_07':'https://images.unsplash.com/photo-1506794778202-cad84cf45f1d?w=300',
        'person_08':'https://images.unsplash.com/photo-1534528741775-53994a69daeb?w=300',
        'person_09':'https://images.unsplash.com/photo-1552058544-f2b08422138a?w=300',
        'person_10':'https://images.unsplash.com/photo-1531746020798-e6953c6e8e04?w=300',
        'person_11':'https://images.unsplash.com/photo-1519085360753-af0119f7cbe7?w=300',
        'person_12':'https://images.unsplash.com/photo-1517841905240-472988babdf9?w=300',
        'person_13':'https://images.unsplash.com/photo-1539571696357-5a69c17a67c6?w=300',
        'person_14':'https://images.unsplash.com/photo-1488426862026-3ee34a7d66df?w=300',
        'person_15':'https://images.unsplash.com/photo-1504257432389-52343af06ae3?w=300',
        'person_16':'https://images.unsplash.com/photo-1560250097-0b93528c311a?w=300',
        'person_17':'https://images.unsplash.com/photo-1573497019940-1c28c88b4f3e?w=300',
        'person_18':'https://images.unsplash.com/photo-1567532939604-b6b5b0db2604?w=300',
        'person_19':'https://images.unsplash.com/photo-1548142813-c348350df52b?w=300',
        'person_20':'https://images.unsplash.com/photo-1573496359142-b8d87734a5a2?w=300',
    }
    cr=[]
    for nm,u in fu.items():
        p=HH/f'{nm}.jpg'
        if not p.exists(): dli(u, p)
        if p.exists():
            try:
                pr=pdb(str(p), bm)
                pn=nm.replace('_',' ').title()
                cr.append({'person':pn, 'breed_1':pr[0]['breed'], 'conf_1':pr[0]['confidence']})
                print(f'  {pn}: {pr[0]["breed"]} ({pr[0]["confidence"]:.1%})')
            except Exception as e:
                print(f'  {nm}: {e}')
    cdf = pd.DataFrame(cr)
    if len(cdf)>0:
        cdf.to_csv(RR/'tables'/'celebrity_dog_matches.csv', index=False)
    del bm; torch.cuda.empty_cache()

# === PLOTS ===
fig,ax = plt.subplots(1,2, figsize=(16,6))
for n in MN:
    bh,ba = None, 0
    for f in HD.glob(f'{n}*_history.json'):
        with open(f) as fp: h=json.load(fp)
        if h['best_val_acc']>ba: bh,ba = h, h['best_val_acc']
    if bh:
        ax[0].plot(bh['epochs'], bh['val_acc'], label=n, lw=2)
        ax[1].plot(bh['epochs'], bh['val_loss'], label=n, lw=2)
ax[0].set_title('Val Accuracy'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].set_title('Val Loss'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout()
plt.savefig(RR/'graphs'/'accuracy_comparison.png', dpi=150)
plt.show()

if AR:
    bn=max(AR, key=lambda k: AR[k]['top1'])
    r=AR[bn]
    cm=confusion_matrix(r['labels'], r['preds'])
    np.fill_diagonal(cm, 0)
    ti=np.argsort(cm.sum(0)+cm.sum(1))[-20:]
    cmf=confusion_matrix(r['labels'], r['preds'])
    plt.figure(figsize=(14,12))
    sns.heatmap(cmf[np.ix_(ti,ti)], annot=True, fmt='d', cmap='Blues',
                xticklabels=[CN[i] for i in ti], yticklabels=[CN[i] for i in ti])
    plt.title(f'Confusion Matrix -- {bn}')
    plt.xlabel('Predicted'); plt.ylabel('Actual')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(RR/'graphs'/'confusion_matrix.png', dpi=150)
    plt.show()

# === RESULTS EXPORT ===
ex = {'model_comparison':{}, 'transfer_learning':{}, 'training_times':{}, 'best_model':''}
for n in MN:
    if n in AR:
        r=AR[n]
        ex['model_comparison'][n] = {'top1':round(r['top1'],4), 'top5':round(r['top5'],4)}
    sg={}; tt=0
    for s in ['stage1','stage2','stage3']:
        hp=HD/f'{n}_{s}_history.json'
        if hp.exists():
            with open(hp) as f: h=json.load(f)
            sg[s]=round(h['best_val_acc'],4)
            tt+=h.get('training_time_seconds',0)
    if sg:
        ex['transfer_learning'][n]=sg
        ex['training_times'][n]=round(tt,1)
if AR:
    ex['best_model'] = max(AR, key=lambda k: AR[k]['top1'])
if len(adf)>0:
    ex['animal_experiment'] = adf.to_dict('records')
if len(cdf)>0:
    ex['celebrity_experiment'] = cdf.to_dict('records')

print('\n=== COPY EVERYTHING BELOW THIS LINE ===')
print(json.dumps(ex, indent=2))
print('=== COPY EVERYTHING ABOVE THIS LINE ===')
